In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import shutil
import random

zip_path = "/content/drive/MyDrive/plant_training/plant_dataset_clean.zip"
!unzip -q "{zip_path}" -d /content/dataset

print("Unzip complete.")

Unzip complete.


In [13]:
import os
import shutil
import random

train_dir = '/content/dataset/train'
val_dir = '/content/dataset/val'

# 1. First, move everything from val back to train to reset
if os.path.exists(val_dir):
    for cls in os.listdir(val_dir):
        val_cls_path = os.path.join(val_dir, cls)
        train_cls_path = os.path.join(train_dir, cls)
        os.makedirs(train_cls_path, exist_ok=True)
        for img in os.listdir(val_cls_path):
            shutil.move(os.path.join(val_cls_path, img), os.path.join(train_cls_path, img))
    shutil.rmtree(val_dir)
    print("Reset complete: All images moved back to train.")

# 2. Now perform the correct split
os.makedirs(val_dir, exist_ok=True)
for cls in os.listdir(train_dir):
    src_path = os.path.join(train_dir, cls)
    if os.path.isdir(src_path):
        dst_path = os.path.join(val_dir, cls)
        os.makedirs(dst_path, exist_ok=True)

        images = os.listdir(src_path)
        random.shuffle(images)

        if len(images) <= 1:
            # COPY if only 1 image exists so both folders are happy
            for img in images:
                shutil.copy(os.path.join(src_path, img), os.path.join(dst_path, img))
            print(f"Mirrored single image for: {cls}")
        else:
            # MOVE 20% to val for normal classes
            val_count = max(1, int(len(images) * 0.2))
            for i in range(val_count):
                shutil.move(os.path.join(src_path, images[i]), os.path.join(dst_path, images[i]))

print("Validation split finished correctly.")

Reset complete: All images moved back to train.
Mirrored single image for: Tomato_two_spotted_spider_mites_leaf
Validation split finished correctly.


In [14]:
from torchvision import datasets, transforms
import torch

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

image_datasets = {x: datasets.ImageFolder(os.path.join('/content/dataset', x), data_transforms[x])
                  for x in ['train', 'val']}

dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=32, shuffle=True, num_workers=2)
              for x in ['train', 'val']}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

print(f"Successfully loaded {len(class_names)} classes.")

Successfully loaded 28 classes.


In [16]:
from torchvision import models
import torch.nn as nn

model_mobile = models.mobilenet_v2(weights='DEFAULT')
num_ftrs = model_mobile.classifier[1].in_features
model_mobile.classifier[1] = nn.Linear(num_ftrs, 28)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model_mobile = model_mobile.to(device)
print("MobileNetV2 ready.")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 154MB/s]


MobileNetV2 ready.


In [18]:
import time
import copy

def train_model(model, criterion, optimizer, scheduler, num_epochs=15):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model

In [19]:
import torch.optim as optim
from torch.optim import lr_scheduler

criterion = nn.CrossEntropyLoss()

# Observe that all parameters are being optimized
optimizer_ft = optim.SGD(model_mobile.parameters(), lr=0.001, momentum=0.9)

# Decay LR by a factor of 0.1 every 7 epochs
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)

# Start training
model_mobile = train_model(model_mobile, criterion, optimizer_ft, exp_lr_scheduler, num_epochs=15)

Epoch 0/14
----------
train Loss: 1.4901 Acc: 0.6247
val Loss: 0.5915 Acc: 0.8490

Epoch 1/14
----------
train Loss: 0.4531 Acc: 0.8825
val Loss: 0.3184 Acc: 0.9156

Epoch 2/14
----------
train Loss: 0.3027 Acc: 0.9143
val Loss: 0.2364 Acc: 0.9331

Epoch 3/14
----------
train Loss: 0.2438 Acc: 0.9283
val Loss: 0.2085 Acc: 0.9387

Epoch 4/14
----------
train Loss: 0.2113 Acc: 0.9386
val Loss: 0.1869 Acc: 0.9447

Epoch 5/14
----------
train Loss: 0.1884 Acc: 0.9428
val Loss: 0.1743 Acc: 0.9473

Epoch 6/14
----------
train Loss: 0.1664 Acc: 0.9516
val Loss: 0.1579 Acc: 0.9521

Epoch 7/14
----------
train Loss: 0.1477 Acc: 0.9571
val Loss: 0.1606 Acc: 0.9498

Epoch 8/14
----------
train Loss: 0.1472 Acc: 0.9565
val Loss: 0.1539 Acc: 0.9533

Epoch 9/14
----------
train Loss: 0.1413 Acc: 0.9596
val Loss: 0.1541 Acc: 0.9523

Epoch 10/14
----------
train Loss: 0.1420 Acc: 0.9567
val Loss: 0.1561 Acc: 0.9521

Epoch 11/14
----------
train Loss: 0.1388 Acc: 0.9595
val Loss: 0.1519 Acc: 0.9539

Ep

In [20]:
torch.save(model_mobile.state_dict(), '/content/drive/MyDrive/plant_disease_mobilenet_v1.pth')

In [21]:
model_mobile.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in dataloaders['val']:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model_mobile(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Final MobileNetV2 Accuracy: {(100 * correct / total):.2f}%')

Final MobileNetV2 Accuracy: 95.39%
